# Provider parity — OpenAI vs DeepSeek

Both providers are used via OpenAI-compatible chat API (DeepSeek through `base_url` override). The enrichment cache is keyed by `(section_id, provider, payload_hash)`, so we get an apples-to-apples comparison.

## 0. Setup

In [ ]:
import sys, os, time
from pathlib import Path
sys.path.insert(0, os.path.abspath(".."))
from src.core.config import settings
from src.ingestion import regex_pass
from src.ingestion.llm_enrich import enrich_section
from src.ingestion.pipeline import load_book_static_metadata

book = load_book_static_metadata("islp")
sections = regex_pass.parse_chapter(
    book_slug="islp", chapter_id="ch02",
    source_path=Path(book.source_path),
    line_start=650, line_end=2557,
)
sample = sections[:3]
print(f"Sampling {len(sample)} sections:")
for s in sample:
    print(" -", s.section_id, "chars=", s.char_count)

## 1. Run both providers

In [ ]:
def run(provider):
    out = []
    for s in sample:
        s_copy = s.model_copy(deep=True)
        s_copy.index_extended = []
        s_copy.synopsis = ""
        t0 = time.perf_counter()
        enriched = enrich_section(s_copy, provider=provider, existing_index=book.index_terms)
        dt = time.perf_counter() - t0
        out.append((enriched, dt))
    return out

openai_results = run("openai")
deepseek_results = run("deepseek")

## 2. Compare synopsis length & timing

In [ ]:
print(f"{'section':30s} {'OAI_chars':>10s} {'DS_chars':>10s} {'OAI_t':>8s} {'DS_t':>8s}")
for (oai, dt_oai), (ds, dt_ds) in zip(openai_results, deepseek_results):
    print(f"{oai.section_id[:30]:30s} {len(oai.synopsis):>10d} {len(ds.synopsis):>10d} {dt_oai:>8.2f} {dt_ds:>8.2f}")

## 3. Index extended — Jaccard overlap

In [ ]:
def jaccard(a, b):
    a, b = set(map(str.lower, a)), set(map(str.lower, b))
    if not a and not b: return 1.0
    return len(a & b) / max(1, len(a | b))

for (oai, _), (ds, _) in zip(openai_results, deepseek_results):
    print(f"{oai.section_id[:30]:30s}  jaccard={jaccard(oai.index_extended, ds.index_extended):.2f}")
    print("  OAI:", oai.index_extended)
    print("  DS :", ds.index_extended)
    print()

## 4. Side-by-side synopsis

In [ ]:
for (oai, _), (ds, _) in zip(openai_results, deepseek_results):
    print("=" * 80)
    print(oai.section_id, "—", oai.h2_path)
    print("\n[OpenAI]:", oai.synopsis)
    print("\n[DeepSeek]:", ds.synopsis)

## 5. Interpretation guide

- Higher Jaccard → providers converge on similar keywords.
- Big synopsis-length gap → one provider is more verbose; tune via prompt.
- Pick provider based on (a) cost, (b) latency, (c) quality on your domain.
- The pipeline supports flipping via `--provider` flag without code changes.